# 📈 Finance Stock Analysis — Six Major U.S. Banks (2006–2016)

This notebook analyzes historical stock performance of **BAC, C, GS, JPM, MS, WFC**:

| Step | Question |
|------|----------|
| 1. Data collection | Can we reliably load 10 years of prices (and stay reproducible offline)? |
| 2. Exploration | How did prices evolve, including the 2008 crisis? |
| 3. Returns | What are the daily / cumulative / normalized returns? |
| 4. Risk | Which bank is riskiest (volatility, Sharpe, drawdown, VaR)? |
| 5. Correlation | Which banks move together? |
| 6. Trends | What do moving averages say (e.g. BAC)? |
| 7. Portfolio | What does a Monte Carlo efficient frontier look like? |

> **How to run:** `pip install -r requirements.txt`, then run all cells. Live data comes from Yahoo Finance; if it is unreachable the notebook falls back to a reproducible synthetic dataset (and tells you which source was used).
>
> All reusable logic lives in `src/` (`data`, `metrics`, `plots`, `portfolio`) — the notebook is the narrative on top. The same analysis runs headless via `python scripts/run_analysis.py`.

## 0. Setup

Imports and **global plot styling first** (so every chart below looks consistent).

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve()))  # repo root → `import src...`

import matplotlib.pyplot as plt

from src import config, data, metrics, plots, portfolio

plots.setup_style()  # one consistent style for all charts
print("Tickers:", config.BANKS)
print("Period :", config.START_DATE, "→", config.END_DATE)

## 1. Data collection

`load_prices()` tries **local cache → Yahoo Finance → synthetic fallback**, in that order, and always reports which source was used. The extractor handles both old and new `yfinance` column layouts, and `clean_prices()` sorts dates and fills small gaps.

In [ ]:
close, source = data.load_prices(
    config.BANKS, config.START_DATE, config.END_DATE,
    cache_path=config.CACHE_FILE, synthetic_fallback=True,
)
print(f"Data source: {source}")
display(data.describe_data(close))
close.head()

## 2. Price exploration

Raw closing prices. The shaded band marks the **2008–2009 financial crisis** — expect a drawdown everywhere, then a slow multi-year recovery.

In [ ]:
plots.plot_price_history(close)
plt.show()

## 3. Returns: daily, normalized, cumulative

- **Daily returns** (`pct_change`) are the basis of all risk math.
- **Normalized prices** (base 100) make banks comparable despite different price levels.
- **Cumulative returns** show the growth of a $1 investment.

In [ ]:
returns = metrics.daily_returns(close)
normalized = metrics.normalized_prices(close)
cumulative = metrics.cumulative_returns(returns)
returns.head()

In [ ]:
plots.plot_normalized(normalized)
plt.show()

plots.plot_cumulative(cumulative)
plt.show()

## 4. Risk analysis

We measure risk four complementary ways:

- **Annualized volatility** — how much daily returns fluctuate (×√252).
- **Sharpe / Sortino ratios** — return per unit of (downside) risk.
- **Maximum drawdown** — worst peak-to-trough loss.
- **VaR (95%)** — daily loss that is exceeded only 5% of the time.

In [ ]:
summary = metrics.summary_table(close, returns, config.RISK_FREE_RATE)
summary.round(4)

In [ ]:
metrics.best_worst_days(returns)

In [ ]:
plots.plot_rolling_volatility(
    metrics.rolling_volatility(returns, config.ROLLING_VOL_WINDOW),
    config.ROLLING_VOL_WINDOW)
plt.show()

plots.plot_drawdown(metrics.drawdown_series(close))
plt.show()

plots.plot_risk_return(summary)
plt.show()

**Reading the charts:** volatility spikes inside the crisis band; the underwater chart shows how deep each bank fell from its peak; the risk/return scatter (bubble size ∝ Sharpe) shows which banks compensated investors for the risk taken.

## 5. Correlation & distributions

Banks in the same sector usually move together — the heatmap quantifies that. The histograms (±3σ guides) show fat tails: extreme single-day moves happen more often than a normal distribution would predict.

In [ ]:
corr = metrics.correlation_matrix(returns)
plots.plot_correlation_heatmap(corr)
plt.show()
corr.round(2)

In [ ]:
plots.plot_return_distributions(returns)
plt.show()

In [ ]:
# Sampled scatter matrix — the full 2,600-row pairplot is too slow to render
plots.plot_pairwise_scatter(returns, sample=500, seed=config.RANDOM_SEED)
plt.show()

## 6. Trend: moving averages (Bank of America)

A **30-day** average tracks the short-term trend, a **50-day** average the longer one. When the short average crosses above (below) the long one, traders call it a *golden cross* (*death cross*).

In [ ]:
bac_ma = metrics.moving_averages(
    close["BAC"], config.MA_SHORT_WINDOW, config.MA_LONG_WINDOW)
plots.plot_moving_averages(bac_ma, "BAC")
plt.show()

## 7. Portfolio simulation (bonus)

Instead of picking one bank, what mix works best? A **Monte Carlo** search over 5,000 random long-only portfolios sketches the efficient frontier and finds the ★ max-Sharpe and ★ min-volatility allocations.

In [ ]:
simulations, best = portfolio.random_portfolios(
    returns, n_portfolios=config.N_MONTE_CARLO_PORTFOLIOS,
    risk_free_rate=config.RISK_FREE_RATE, seed=config.RANDOM_SEED)
plots.plot_efficient_frontier(simulations)
plt.show()

print("Max-Sharpe weights:")
print(best.max_sharpe_weights.round(4))
print(f"Return {best.max_sharpe_return:.2%} | Vol {best.max_sharpe_vol:.2%} | "
      f"Sharpe {best.max_sharpe_ratio:.2f}")

## 8. Key findings & next steps

Re-run the cell below after any change — it derives the findings from the data, so the text never goes stale.

In [ ]:
riskiest = summary["Ann. Volatility"].idxmax()
calmest = summary["Ann. Volatility"].idxmin()
tmp = corr.copy()
for t in tmp.columns:
    tmp.loc[t, t] = float("-inf")
top = tmp.max().idxmax()
pair = (top, tmp.idxmax()[top])
print(f"• Highest volatility : {riskiest} ({summary.loc[riskiest, 'Ann. Volatility']:.2%})")
print(f"• Lowest volatility  : {calmest} ({summary.loc[calmest, 'Ann. Volatility']:.2%})")
print(f"• Strongest link     : {pair[0]}–{pair[1]} (corr {corr.loc[pair[0], pair[1]]:.2f})")
print(f"• Deepest drawdown   : {summary['Max Drawdown'].idxmin()} "
      f"({summary['Max Drawdown'].min():.2%})")
print("• Crisis effect      : volatility spikes 2008–2009 in the rolling-vol chart")

**Next steps** (see README): Sharpe-driven portfolio optimization with constraints, CAPM beta vs. the S&P 500, regime detection, and ML-based forecasting — all easy to add now that the code is modular and tested (`pytest tests/`).